##DimUser

In [0]:
from pyspark.sql.functions import*
from pyspark.sql.types import*

import os
import sys

project_path=os.path.join(os.getcwd(),'..','..')

sys.path.append(project_path)

from utils.transformation import reusable

Batch loading

In [0]:
df= spark.read.format("parquet").load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/DimUser")

In [0]:
df.display()

##Auto Loader
Spark structure streaming

In [0]:
df_user=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/checkpoint")\
            .load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/DimUser")

In [0]:
display(df_user, checkpointLocation = "abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/display_checkpoint")

###Spark Transformation on DimUser

In [0]:
df_user=df_user.withColumn("user_name",upper(col('user_name')))


In [0]:
df_user_obj=reusable()

df_user=df_user_obj.dropColumn(df_user,['_rescued_data'])
df_user==df_user.dropDuplicates(['user_id'])

In [0]:
display(df_user, checkpointLocation = "abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/display_checkpoint")

In [0]:
display(df_user, checkpointLocation = "abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/display_checkpoint")

##Wrting the data into DElta format

In [0]:
df_user.writeStream.format('delta')\
  .outputMode('append')\
  .option("checkpointLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/checkpoint")\
  .trigger(once=True)\
  .option("path","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimUser/data")\
  .toTable("spotify_cata.silver.DimUser")
   
  



##Dim Artisit


In [0]:
df_art=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimArtist/checkpoint")\
            .load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj=reusable()

df_art=df_art_obj.dropColumn(df_art,['_rescued_data'])
df_art==df_art.dropDuplicates(['artist_id'])

In [0]:
df_art.writeStream.format('delta')\
  .outputMode('append')\
  .option("checkpointLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimArtist/checkpoint")\
  .trigger(once=True)\
  .option("path","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimArtist/data")\
  .toTable("spotify_cata.silver.DimArtist")
   
  



##Dim Track

In [0]:
df_track=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimTrack/checkpoint")\
            .load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/DimTrack")

###Transformation

In [0]:
df_track=df_track.withColumn("durationFlag",when(col("duration_sec")<150,"low")\
                                            .when(col("duration_sec")<300,"medium")\
                                            .otherwise("high"))
                                
df_track=df_track.withColumn("track_name",regexp_replace(col("track_name"),'-',' '))

df_track=reusable().dropColumn(df_track,['_rescued_data'])


In [0]:
df_track.writeStream.format('delta')\
  .outputMode('append')\
  .option("checkpointLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimTrack/checkpoint")\
  .trigger(once=True)\
  .option("path","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimTrack/data")\
  .toTable("spotify_cata.silver.DimTrack")
   

###Dim Date


In [0]:
df_date=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimDate/checkpoint")\
            .load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/DimDate")

In [0]:
df_date=reusable().dropColumn(df_date,['_rescued_data'])

In [0]:
df_date.writeStream.format('delta')\
  .outputMode('append')\
  .option("checkpointLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimDate/checkpoint")\
  .trigger(once=True)\
  .option("path","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/DimDate/data")\
  .toTable("spotify_cata.silver.DimDate")
   

###Fact Stream


In [0]:
df_factStream=spark.readStream.format("cloudFiles")\
    .option("cloudFiles.format","parquet")\
        .option("cloudFiles.schemaLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/FactStream/checkpoint")\
            .load("abfss://bronze@spotifystorageaccountru.dfs.core.windows.net/FactStream")

In [0]:
df_factStream=reusable().dropColumn(df_factStream,['_rescued_data'])

In [0]:
df_factStream.writeStream.format('delta')\
  .outputMode('append')\
  .option("checkpointLocation","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/FactStream/checkpoint")\
  .trigger(once=True)\
  .option("path","abfss://silver@spotifystorageaccountru.dfs.core.windows.net/FactStream/data")\
  .toTable("spotify_cata.silver.FactStream")
   